# Import Libraries

In [1]:
import torch
from torchvision import transforms as T
from utils.set_seed import set_random_seed
from torch.utils.tensorboard import SummaryWriter

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')

In [2]:
print(device)

mps


In [3]:
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet

from data.data import get_dataloaders
import utils.trainer as trainer

# Set Hyperparameters, Load Dataset

In [4]:
# Hyperparameters
# TODO: experiment with different hyperparameters
batch_size = 8
learning_rate = 0.001
epochs = 10

set_random_seed(42) # seed for reproducibility

In [5]:
# Load BDD100KPlus dataset
trainloader, testloader = get_dataloaders(dataset_name="Bdd100kPlus", batch_size=batch_size)

In [6]:
# Print dataset statistics
print(f"Number of training samples: {len(trainloader.dataset)}")
print(f"Number of test samples: {len(testloader.dataset)}")

# print batches
print(f"Number of batches in training set: {len(trainloader)}")
print(f"Number of batches in test set: {len(testloader)}")

Number of training samples: 800
Number of test samples: 200
Number of batches in training set: 100
Number of batches in test set: 25


# Initialize WeatherNet Model

In [7]:
# wn_model: WeatherNet = WeatherNet()
# wn_model: WeatherNetPlusPlus = WeatherNetPlusPlus()
wn_model: MtlWeatherNet = MtlWeatherNet()

# Print the model architecture
print(wn_model)

# Define optimizer for the model
# Could explore SGD, Adam, AdamW, etc
optimizer = torch.optim.Adam(wn_model.parameters())

# Define loss function
criterion = torch.nn.CrossEntropyLoss()

MtlWeatherNet(
  (backbone_model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequentia

In [8]:
saved_state = None # path to saved model state
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    wn_model.load_state_dict(torch.load(saved_state, weights_only=True))

# Model Training

In [9]:
# Train the WeatherNet model and record losses
# Use the train function to train the wn_model with optimizer on the trainloader for a specified number of epochs (e.g., 5)
# Record the training and test losses in wn_train_losses and wn_test_losses respectively, pass hyperparameters

# logs to runs/WeatherNet/ or runs/WeatherNetPlusPlus/ or runs/MtlWeatherNet/
writer = SummaryWriter(log_dir=f"runs/{wn_model.name}")

# epochs=1
wn_train_loss_log, wn_test_loss_log = trainer.train(wn_model, optimizer, criterion, trainloader, testloader, epochs, device, "./checkpoints", writer=writer)

writer.flush()

Begin training
Epoch 1/10
* Batch 10/100 - night loss: 0.59 | glare loss: 0.18 | weather loss: 1.16 | fog loss: 0.09 | road loss: 0.47 | traffic loss: 0.62 | scene loss: 0.88
* Batch 20/100 - night loss: 0.88 | glare loss: 0.44 | weather loss: 1.50 | fog loss: 0.05 | road loss: 0.14 | traffic loss: 0.81 | scene loss: 0.66
* Batch 30/100 - night loss: 0.13 | glare loss: 0.57 | weather loss: 1.06 | fog loss: 0.03 | road loss: 0.06 | traffic loss: 0.92 | scene loss: 0.66
* Batch 40/100 - night loss: 0.08 | glare loss: 0.30 | weather loss: 1.20 | fog loss: 0.01 | road loss: 0.02 | traffic loss: 0.77 | scene loss: 1.07
* Batch 50/100 - night loss: 0.30 | glare loss: 0.56 | weather loss: 1.17 | fog loss: 0.01 | road loss: 0.35 | traffic loss: 1.13 | scene loss: 0.69
* Batch 60/100 - night loss: 0.08 | glare loss: 0.10 | weather loss: 1.25 | fog loss: 0.00 | road loss: 0.06 | traffic loss: 0.84 | scene loss: 0.82
* Batch 70/100 - night loss: 0.19 | glare loss: 0.14 | weather loss: 1.58 | fog 

# Visualize Results

First, ensure you're in the correct environment:

`conda env create -f environment.yml`

This will create a conda environment called *tensorboard*, which you can activate via `conda activate tensorboard`

Then, to visualize the results: 

`tensorboard --logdir=runs`

This will automatically and recursively scan through all run logs in the runs/ directory.